# 07 · Una red neuronal que lee dígitos (y se equivoca)
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andrezconz/investigacion-aumentada-ml/blob/main/notebooks/07_red_neuronal_digitos.ipynb)

**El dato:** 1.797 dígitos escritos a mano por personas reales, digitalizados en imágenes de 8 × 8 píxeles (conjunto *Optical Recognition of Handwritten Digits*, UCI). Viene incluido en scikit-learn: no hay que descargar nada.

**Familia:** supervisado. Cada imagen trae su etiqueta: el dígito que la persona quiso escribir.

**Cómo funciona una red neuronal:** la imagen entra como 64 números (uno por píxel), pasa por una capa de neuronas que combinan esos números con pesos, y sale una probabilidad para cada dígito. Los pesos se ajustan con cada ejemplo hasta acertar.

In [ ]:
from sklearn.datasets import load_digits
import matplotlib.pyplot as plt
d = load_digits()
print(d.images.shape[0], "imágenes de", d.images.shape[1], "×", d.images.shape[2], "píxeles")
fig, axs = plt.subplots(1, 10, figsize=(12, 1.8))
for ax, img, y in zip(axs, d.images, d.target):
    ax.imshow(img, cmap="gray_r"); ax.set_title(y); ax.axis("off")
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

X_ent, X_prueba, y_ent, y_prueba, img_ent, img_prueba = train_test_split(
    d.data / 16, d.target, d.images, test_size=0.3, random_state=42, stratify=d.target)

red = MLPClassifier(hidden_layer_sizes=(40,), alpha=0.01, max_iter=2000, random_state=0).fit(X_ent, y_ent)
print("Aciertos con lo que vio:   ", round(red.score(X_ent, y_ent) * 100, 1), "%")
print("Aciertos con lo que no vio:", round(red.score(X_prueba, y_prueba) * 100, 1), "%")

La distancia entre los dos números es **sobreajuste**: una parte de lo que la red «sabe» es memoria de los ejemplos que vio.

## Mirar los errores uno por uno

In [ ]:
import numpy as np, pandas as pd
pred = red.predict(X_prueba)
errores = np.where(pred != y_prueba)[0]
print("Errores:", len(errores), "de", len(y_prueba))

pares = pd.Series([f"{a} → {b}" for a, b in zip(y_prueba[errores], pred[errores])]).value_counts().head(5)
print("Confusiones más frecuentes (real → dijo el modelo):"); print(pares.to_string())

n = min(12, len(errores))
fig, axs = plt.subplots(2, 6, figsize=(12, 4.6))
for ax, i in zip(axs.flat, errores[:n]):
    ax.imshow(img_prueba[i], cmap="gray_r"); ax.set_title(f"Real {y_prueba[i]} · dijo {pred[i]}"); ax.axis("off")
plt.tight_layout(); plt.show()

> **Tip:** mira los errores uno por uno. Dicen más que el porcentaje y muestran qué aprendió de verdad el modelo. Varios de estos dígitos serían ambiguos incluso para una persona.

*Fuente: Alpaydin y Kaynak, Optical Recognition of Handwritten Digits, UCI Machine Learning Repository (incluido en scikit-learn). Elaboración propia.*